# CIFAR-10 Image Classification: Advanced Deep Learning Techniques

**MSc Data Science - DSM150 Neural Networks - Final Coursework**

---

## Abstract

This report investigates advanced deep learning techniques for CIFAR-10 image classification. Using a champion-challenger methodology, we progress from a simple ConvNet baseline (~65% accuracy) through data augmentation, residual architectures, and transfer learning, achieving ~95.7% test accuracy with a fine-tuned ResNet50V2.

**Reproducibility**: Global seed = 128 (`tf.keras.utils.set_random_seed(128)`). Long training runs are gated behind `if False:`. Pre-computed results are in `training_histories/` and `evaluations/`; model weights in `saved_models/`. **Environment**: Python 3.10+, TensorFlow 2.x, Keras 3.x.


---

# 1 | Problem definition, dataset description and data preparation

## 1.1 | Task Specification

The task is *multi-class image classification*: assign one of 10 labels to each 32×32 colour image. The **CIFAR-10** dataset (Krizhevsky & Hinton, 2009) contains 60k images across 10 balanced classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck): 50k training / 10k test, each 32×32×3 pixels, pixel values 0–255.


## 1.2 Data Import and Basic Inspection

We begin by importing necessary libraries and loading the CIFAR-10 dataset directly from Keras datasets.

In [ ]:
### import required libraries

# primary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import pandas as pd
import os

# kearas
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# additional helpers
from sklearn.model_selection import train_test_split

# set random seeds for reproducibility
np.random.seed(128)
tf.random.set_seed(128)

# check TensorFlow version and GPU availability
print(f"tensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")
print(f"keras version: {keras.__version__}")

# test global data type
# mixed precision for speedup on compatible hardware (e.g., NVIDIA GPUs with Tensor Cores)
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print(f"global data type: {tf.keras.mixed_precision.global_policy()}")

In [ ]:
# load CIFAR-10 dataset
(X_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()

# define class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

# display dataset shapes
print("Dataset Shapes:")
print(f"Training images: {X_train_full.shape}")
print(f"Training labels: {y_train_full.shape}")
print(f"Test images: {X_test.shape}")
print(f"Test labels: {y_test.shape}")
print('\n')
print(f"Image dimensions: {X_train_full.shape[1:]}")
print(f"Number of classes: {len(class_names)}")
print(f"Pixel value range: [{X_train_full.min()}, {X_train_full.max()}]")

In [ ]:
# visualize sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Figure 1:Sample Images from Each CIFAR-10 Class', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    # find first occurrence of each class
    idx = np.where(y_train_full == i)[0][0]
    # display image and class name
    ax.imshow(X_train_full[idx])
    ax.set_title(f"{class_names[i]}\n(Class {i})", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# analyze class distribution
train_class_counts = np.bincount(y_train_full.flatten())
test_class_counts = np.bincount(y_test.flatten())

# plot class distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# training set distribution
ax1.bar(range(10), train_class_counts, color='steelblue', alpha=0.8)
ax1.set_xlabel('Class', fontsize=11)
ax1.set_ylabel('Number of Images', fontsize=11)
ax1.set_title('Training Set Class Distribution', fontsize=12, fontweight='bold')
ax1.set_xticks(range(10))
ax1.set_xticklabels(class_names, rotation=45, ha='right')
ax1.grid(axis='y', alpha=0.3)

# test set distribution
ax2.bar(range(10), test_class_counts, color='coral', alpha=0.8)
ax2.set_xlabel('Class', fontsize=11)
ax2.set_ylabel('Number of Images', fontsize=11)
ax2.set_title('Test Set Class Distribution', fontsize=12, fontweight='bold')
ax2.set_xticks(range(10))
ax2.set_xticklabels(class_names, rotation=45, ha='right')
ax2.grid(axis='y', alpha=0.3)

# title
fig.suptitle('Figure 2: Class Distribution in CIFAR-10 Dataset', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nclass distribution summary:")
print(f"Training set: {train_class_counts[0]} images per class (balanced)")
print(f"Test set: {test_class_counts[0]} images per class (balanced)")

## 1.3 Data Preprocessing

Pixel values are min-max normalised to [0, 1] ($x_{norm} = x / 255$) for stable gradient flow. 10% of training data (5,000 images) is held out as a validation set; the test set is only used for final evaluation.


In [ ]:
# normalize pixel values to [0, 1] range
# swap to mixed-precision dtype for optimization on GPU
X_train_full_16 = X_train_full.astype('float16') / 255.0
X_test_16 = X_test.astype('float16') / 255.0

X_train_full = X_train_full.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# create validation set (10% of training data)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full_16, y_train_full, 
    test_size=0.1, 
    random_state=128, 
    stratify=y_train_full
)

print("Data Split Summary:")
print(f"Training set: {X_train.shape[0]} images")
print(f"Validation set: {X_val.shape[0]} images")
print(f"Test set: {X_test_16.shape[0]} images")
print(f"\nNormalized pixel value range: [{X_train.min():.2f}, {X_train.max():.2f}]")

### One-Hot Encoding

Labels are one-hot encoded (e.g. class 3 → `[0,0,0,1,0,0,0,0,0,0]`) to enable softmax output and categorical cross-entropy loss.


In [ ]:
# one-hot encode labels
y_train_encoded = to_categorical(y_train, 10)
y_val_encoded = to_categorical(y_val, 10)
y_test_encoded = to_categorical(y_test, 10)

print("Label Encoding:")
print(f"Original label shape: {y_train.shape}")
print(f"Encoded label shape: {y_train_encoded.shape}")
print(f"\nExample encoding for class {y_train[0][0]}:")
print(f"Original: {y_train[0][0]}")
print(f"One-hot: {y_train_encoded[0]}")

### Efficient Data Loading

A `tf.data` pipeline with prefetching is used to reduce CPU/GPU bottlenecks during training.


## 1.4 Data Augmentation Pipeline

Data augmentation artificially expands the training set, reducing overfitting and improving generalisation. We apply: horizontal flipping (p=0.5), width/height shifts (±10%), zoom (±10%), and small rotations (±10°). Each transformation preserves class identity while adding variation the model must learn to be invariant to.

Two augmentation backends were evaluated: CPU-bound `ImageDataGenerator` and a custom GPU `RandomAffineIDGLike` Keras layer. The CPU method was preferred for image quality (see §1.4 note and Appendix A).


In [ ]:
# seed
tf.keras.utils.set_random_seed(128)

# GPU data augmentation using ImageDataGenerator (CPU bound, but better for small images)
data_augmentation = ImageDataGenerator(
    rotation_range=15,           # Random rotations up to 15 degrees
    width_shift_range=0.1,       # Horizontal shifts up to 10%
    height_shift_range=0.1,      # Vertical shifts up to 10%
    horizontal_flip=True,        # Random horizontal flips
    zoom_range=0.1,              # Random zoom up to 10%
    fill_mode='nearest',         # Fill strategy for pixels outside boundaries
    interpolation_order=1,       # Use bilinear interpolation (order=1) for better quality
)

print("Data Augmentation Configuration:")
print(f"Rotation range: +-15 degrees")
print(f"Width shift range: +-10%")
print(f"Height shift range: +-10%")
print(f"Horizontal flip: Enabled")
print(f"Zoom range: +-10%")
print(f"Fill mode: Nearest")

In [ ]:
# visualize augmentation effects
fig, axes = plt.subplots(3, 6, figsize=(15, 7))
fig.suptitle(
    'Figure 3: Data Augmentation Examples',
    fontsize=14,
    fontweight='bold'
)
    
# select 3 random images from training set
sample_indices = np.random.choice(len(X_train_full), size=3, replace=False)

for row, idx in enumerate(sample_indices):
    original_img = X_train_full[idx]

    # Show original
    axes[row, 0].imshow(original_img)
    axes[row, 0].set_title('Original', fontsize=10)
    axes[row, 0].axis('off')

    # Generate 5 augmented versions using generator
    for col in range(1, 6):
        aug_iter = data_augmentation.flow(
            np.expand_dims(original_img, axis=0), 
            batch_size=1, 
            shuffle=False
        )
        aug_img = next(aug_iter)[0]

        axes[row, col].imshow(aug_img)
        axes[row, col].set_title(f'Augmented {col}', fontsize=10)
        axes[row, col].axis('off')

print("The figure above demonstrates how data augmentation creates diverse training examples")
print("from a single image, helping the model learn rotation, translation, zoom, and flip invariances.")

> **Custom augmentation layer**: `RandomAffineIDGLike` applies a composite affine transform in a single GPU operation, matching the output quality of `ImageDataGenerator` while eliminating the CPU bottleneck. See `data_aug_testing.ipynb` for a visual comparison.


In [ ]:
import math
# a little bit of linear algebra as a treat

# custom augmentation layer for GPU-accelerated affine transformations
class RandomAffineIDGLike(tf.keras.layers.Layer):
    """Single-resample affine aug: rot+-15 deg, shift+-10%, zoom [0.9,1.1], hflip, nearest fill, bilinear interp."""
    def __init__(self, seed=128, **kwargs):
        super().__init__(**kwargs)
        self.rng = tf.random.Generator.from_seed(seed)

    # check if training is True, then apply augmentation;
    # otherwise return images unchanged
    def call(self, images, training=None):
        if training is not True:
            return images

        x_in = tf.convert_to_tensor(images)
        in_dtype = x_in.dtype

        # always run the transform in float32 for kernel support & stable math
        x = tf.cast(x_in, tf.float32)

        # generate random parameters for each image in the batch
        b = tf.shape(x)[0]
        h = tf.cast(tf.shape(x)[1], tf.float32)
        w = tf.cast(tf.shape(x)[2], tf.float32)

        rot = self.rng.uniform([b], -15.0, 15.0) * (math.pi / 180.0)
        tx  = self.rng.uniform([b], -0.1, 0.1) * w
        ty  = self.rng.uniform([b], -0.1, 0.1) * h
        z   = self.rng.uniform([b], 0.9, 1.1)

        # random horizontal flip by negating the x scale factor
        flip = self.rng.uniform([b], 0.0, 1.0) < 0.5
        sx = tf.where(flip, -z, z)
        sy = z

        # compute combined affine transform matrix components
        cr = tf.cos(rot); sr = tf.sin(rot)
        cx = (w - 1.0) / 2.0
        cy = (h - 1.0) / 2.0

        # compute the forward affine transform components
        a0 = sx * cr
        a1 = -sx * sr
        b0 = sy * sr
        b1 = sy * cr
        a2 = cx - a0 * cx - a1 * cy + tx
        b2 = cy - b0 * cx - b1 * cy + ty

        # invert the affine transform for use with image_projective_transform
        det = a0 * b1 - a1 * b0
        ia0 =  b1 / det
        ia1 = -a1 / det
        ib0 = -b0 / det
        ib1 =  a0 / det
        ia2 = -(ia0 * a2 + ia1 * b2)
        ib2 = -(ib0 * a2 + ib1 * b2)

        # stack the transforms into the shape required by image_projective_transform
        t = tf.stack(
            [ia0, ia1, ia2, ib0, ib1, ib2, tf.zeros([b]), tf.zeros([b])],
            axis=1
        )

        # apply the projective transform to the batch of images
        y = tf.raw_ops.ImageProjectiveTransformV3(
            images=x,
            transforms=t,
            output_shape=tf.cast(tf.stack([h, w]), tf.int32),
            interpolation="BILINEAR",
            fill_mode="NEAREST",
            fill_value=tf.constant(0.0, dtype=tf.float32),
        )

        # cast back so downstream layers run in mixed precision as intended
        return tf.cast(y, in_dtype)

---

# 2 | Model building, training and evaluation

## 2.1 | Baseline Model Purpose

A baseline provides the performance floor against which all improvements are measured. A random classifier scores ~10% on CIFAR-10; a simple convolutional baseline should comfortably exceed this while exposing the fundamental limitations (overfitting, capacity ceiling) that motivate the systematic improvements in Section 3.

Our baseline follows the classical LeNet-style pattern: convolutional blocks for hierarchical feature extraction, max pooling for spatial subsampling and translation invariance, and dense layers for classification — embedding core deep learning principles in a minimal and interpretable architecture.


## 2.2 The Basic ConvNet Architecture

Two convolutional blocks followed by dense layers:

- **Block 1**: Conv2D(32, 3×3) → ReLU → MaxPool(2×2)
- **Block 2**: Conv2D(64, 3×3) → ReLU → MaxPool(2×2)
- **Head**: Flatten → Dense(32) → ReLU → Dense(10) → Softmax

Spatial dimensions reduce 32×32 → 16×16 → 8×8 while feature maps deepen (32→64), following the principle that deeper layers form increasingly abstract representations. 3×3 kernels capture local patterns; max pooling provides translation invariance and reduces parameters for downstream layers.


In [ ]:
# seed
tf.keras.utils.set_random_seed(128)

def create_baseline_model():
    # create baseline model
    baseline_model = models.Sequential(name='Baseline_ConvNet')

    # input
    baseline_model.add(layers.Input(shape=(32, 32, 3), name='input_layer'))

    # block 1
    baseline_model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same', name='conv1'))
    baseline_model.add(layers.MaxPooling2D((2, 2), name='pool1'))

    # block 2
    baseline_model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2'))
    baseline_model.add(layers.MaxPooling2D((2, 2), name='pool2'))

    # classifier head
    baseline_model.add(layers.Flatten(name='flatten'))
    baseline_model.add(layers.Dense(32, activation='relu', name='dense1'))
    baseline_model.add(layers.Dense(10, activation='softmax', name='output'))

    # compile model
    baseline_model.compile(
        optimizer='RMSprop',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return baseline_model


# create model
baseline_model = create_baseline_model()


# display model architecture and parameter count
baseline_model.summary()

### Training Configuration

- **Optimizer**: RMSprop | **Loss**: Categorical cross-entropy
- **Batch size**: 128 | **Epochs**: 50 | **No callbacks or regularization**

This minimal setup reveals the model's natural learning dynamics as a clear baseline for comparison.


In [ ]:
# NOTE: Training is gated behind `if False:` to prevent accidental re-runs.
# To re-train from scratch, change `if False:` to `if True:` or remove the guard.
# Pre-computed histories are loaded in the cells below.
if False:
    # seed
    tf.keras.utils.set_random_seed(128)
    # train baseline model
    print("Training baseline model ...")
    print("Configuration: RMSprop optimizer, batch_size=128, epochs=50")
    
    history_baseline = baseline_model.fit(
        X_train, y_train_encoded,
        batch_size=128,
        epochs=50,
        validation_data=(X_val, y_val_encoded),
        verbose=1
    )

In [ ]:
# helper function to plot training history
def plot_training_history(history, title='Figure 4: Model Training History'):
    # create empty plot with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    if isinstance(history, dict):
        history_data = history
    else:
        history_data = history.history
    
    # plot accuracy
    ax1.plot(history_data['accuracy'], label='Training Accuracy', linewidth=2)
    ax1.plot(history_data['val_accuracy'], label='Validation Accuracy', linewidth=2)
    ax1.set_yticks(np.arange(0, 1, 0.05))
    ax1.set_xlabel('Epoch', fontsize=11)
    ax1.set_ylabel('Accuracy', fontsize=11)
    ax1.set_title('Model Accuracy', fontsize=12, fontweight='bold')
    ax1.legend(loc='lower right')
    ax1.grid(alpha=0.3)
    
    # plot loss
    ax2.plot(history_data['loss'], label='Training Loss', linewidth=2)
    ax2.plot(history_data['val_loss'], label='Validation Loss', linewidth=2)
    ax2.set_yticks(np.arange(0, max(history_data['loss']) + 0.5, 0.5))
    ax2.set_xlabel('Epoch', fontsize=11)
    ax2.set_ylabel('Loss', fontsize=11)
    ax2.set_title('Model Loss', fontsize=12, fontweight='bold')
    ax2.legend(loc='upper right')
    ax2.grid(alpha=0.3)
    
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# plot baseline training history
plot_training_history(history_baseline, 'Figure 3: Baseline ConvNet Training History')

## 2.3 Baseline Results and Analysis

### Performance Evaluation

In [ ]:
# Evaluate baseline model on test set
test_loss_baseline, test_accuracy_baseline = baseline_model.evaluate(X_test, y_test_encoded, verbose=0)

print("="*60)
print("BASELINE MODEL PERFORMANCE")
print("="*60)
print(f"Test Accuracy: {test_accuracy_baseline*100:.2f}%")
print(f"Test Loss: {test_loss_baseline:.4f}")
print(f"\nFinal Training Accuracy: {history_baseline.history['accuracy'][-1]*100:.2f}%")
print(f"Final Validation Accuracy: {history_baseline.history['val_accuracy'][-1]*100:.2f}%")
print(f"\nBest Validation Accuracy: {max(history_baseline.history['val_accuracy'])*100:.2f}%")
print(f"Best Validation Epoch: {np.argmax(history_baseline.history['val_accuracy']) + 1}")
print("="*60)

### Key Observations

A train/validation accuracy gap of ~18% confirms significant overfitting. Validation peaks near epoch 16 (~70%) then plateaus, suggesting the model has exhausted its capacity. Final test accuracy is ~65%, a ~17.5% improvement over the midterm fully-connected baseline (47.5%), confirming that convolutional inductive biases (weight sharing, spatial locality, translation invariance) are highly effective for image data.

This establishes a performance ceiling and motivates Section 3 improvements across three axes: (1) regularisation and effective dataset size via augmentation, (2) optimisation dynamics via callbacks, and (3) representational capacity via advanced architecture.


---

# 3 | Model improvement – without using pretrained models

## 3.1 Champion-Challenger Methodology

The champion-challenger framework introduces exactly one change at a time, comparing against the current best model. Changes are retained only if they improve held-out performance, ensuring every design decision is evidence-based and regressions are prevented. This enables clear attribution of performance gains to individual modifications.

Three challengers are evaluated in a dependency-ordered sequence:
1. **Data augmentation + architectural capacity** — foundational: augmentation enables the larger models that follow
2. **Training callbacks** — optimise learning dynamics on top of the augmented champion
3. **ResNet-style architecture + BN** — leverage the regularised training regime established above


## 3.2 Challenger 1: Data Augmentation and Architectural Capacity

**Motivation**: The baseline overfits substantially (~18% train/val gap). Data augmentation regularises by presenting varied image transforms each epoch, forcing the model to learn robust invariances rather than memorising specific examples. A deeper, wider architecture can then leverage this enlarged effective dataset. Larger batch sizes (128–1024) are tested: larger batches provide more accurate gradient estimates, which may be particularly beneficial when training on highly diverse augmented data.

**Architecture** (3 blocks + progressive dropout):
- Block 1: Conv2D(64)×3 → stride-2 Conv → Dropout(0.2)
- Block 2: Conv2D(128)×3 → stride-2 Conv → Dropout(0.3)
- Block 3: Conv2D(256)×3 → stride-2 Conv → Dropout(0.4)
- Head: Flatten → Dense(512) → Dropout(0.5) → Dense(10, Softmax)

Strided convolutions replace max pooling for learnable downsampling (Springenberg et al., 2014). Stacked 3×3 convolutions achieve a 5×5 receptive field with fewer parameters and added non-linearity (Simonyan & Zisserman, 2014). Progressive dropout rates (0.2→0.5) increase regularisation at deeper, more abstract levels of the hierarchy.


In [ ]:
def create_augmented_model(seed=128, name='Augmented_ConvNet'):
    """
    create a ConvNet with specified architecture below.
    
    Architecture:
    - Block 1: Conv2D(3x3, 64) -> ReLU -> Conv2D(3x3, 64) -> ReLU -> Conv2D(3x3, 64, stride=2) -> Dropout(0.2)
    - Block 2: Conv2D(3x3, 128) -> ReLU -> Conv2D(3x3, 128) -> ReLU -> Conv2D(3x3, 128, stride=2) -> Dropout(0.3)
    - Block 3: Conv2D(3x3, 256) -> ReLU -> Conv2D(3x3, 256) -> ReLU -> Conv2D(3x3, 256, stride=2) -> Dropout(0.4)
    - Classifier: Flatten -> Dense(512) -> Dropout(0.5) -> Dense(10) -> Softmax

    Args:
        seed: Random seed for reproducibility, default 128
        name: Name of the model, default 'Augmented_ConvNet'
    
    Returns:
        complete, compiled model
    """
    # seed
    tf.keras.utils.set_random_seed(seed)
    
    model = models.Sequential([

        # input layer
        layers.Input(shape=(32, 32, 3), name='input_layer'),
        

        # block 1, 64 filters
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv1_1'),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv1_2'),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', strides=(2, 2), name='conv1_3'),
        layers.Dropout(0.2, name='dropout1'),
        
        # block 2, 128 filters
        layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv2_1'),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv2_2'),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same', strides=(2, 2), name='conv2_3'),
        layers.Dropout(0.3, name='dropout2'),
        
        # block 3, 256 filters
        layers.Conv2D(256, (3, 3), activation='relu', padding='same', name='conv3_1'),
        layers.Conv2D(256, (3, 3), activation='relu', padding='same', name='conv3_2'),
        layers.Conv2D(256, (3, 3), activation='relu', padding='same', strides=(2, 2), name='conv3_3'),
        layers.Dropout(0.4, name='dropout3'),
        
        # classifier head
        layers.Flatten(name='flatten'),
        layers.Dense(512, activation='relu', name='dense1'),
        layers.Dropout(0.5, name='dropout4'),
        layers.Dense(10, activation='softmax', name='output')
    ], name=name)
    
    
    model.compile(
        optimizer='RMSprop',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [ ]:
# create 128 batch model
augmented_model_128 = create_augmented_model(seed=128, name='Augmented_ConvNet_Batch128')
# creating 256 batch
augmented_model_256 = create_augmented_model(seed=128, name='Augmented_ConvNet_Batch256')
# creating 384 batch
augmented_model_384 = create_augmented_model(seed=128, name='Augmented_ConvNet_Batch384')
# creating 512 batch
augmented_model_512 = create_augmented_model(seed=128, name='Augmented_ConvNet_Batch512')
# creating 1024 batch
augmented_model_1024 = create_augmented_model(seed=128, name='Augmented_ConvNet_Batch1024')

# example summary
augmented_model_256.summary()

In [ ]:
# seed
tf.keras.utils.set_random_seed(128)

# Note: The training code is wrapped in an `if False:` block to prevent accidental execution during development.
if False:

    # train with both augmentation methods
    print("Training Challenger 1: Augmented Data + Increased Capacity")
    print("Using ImageDataGenerator for real-time augmentation\n")

    # define number of epochs
    epochs = 300

    # training 128 batch
    history_augmented_128 = augmented_model_128.fit(
        data_augmentation.flow(X_train, y_train_encoded, batch_size=128),
        epochs=epochs,
        validation_data=(X_val, y_val_encoded),
        verbose=1 
    )

    # training 256 batch
    history_augmented_256 = augmented_model_256.fit(
        data_augmentation.flow(X_train, y_train_encoded, batch_size=256),
        epochs=epochs,
        validation_data=(X_val, y_val_encoded),
        verbose=1
    )

    # training 384 batch
    history_augmented_384 = augmented_model_384.fit(
        data_augmentation.flow(X_train, y_train_encoded, batch_size=384),
        epochs=epochs,
        validation_data=(X_val, y_val_encoded),
        verbose=1
    )

    # training 512 batch
    history_augmented_512 = augmented_model_512.fit(
        data_augmentation.flow(X_train, y_train_encoded, batch_size=512),
        epochs=epochs,
        validation_data=(X_val, y_val_encoded),
        verbose=1
    )

    # training 1024 batch
    history_augmented_1024 = augmented_model_1024.fit(
        data_augmentation.flow(X_train, y_train_encoded, batch_size=1024),
        epochs=epochs,
        validation_data=(X_val, y_val_encoded),
        verbose=1
    )

    # store histories in a dictionary for easy access
    augmented_histories = {'Batch 128': history_augmented_128,
                        'Batch 256': history_augmented_256,
                        'Batch 384': history_augmented_384,
                        'Batch 512': history_augmented_512,
                        'Batch 1024': history_augmented_1024}

In [ ]:
# only run if training was executed
if False:
    # save histories
    for batch_size, history in augmented_histories.items():
        np.save(f'training_histories/augmented_history_{batch_size.replace(" ", "_")}.npy', history.history)

    # save models
    augmented_model_128.save('saved_models/augmented_model_batch_128.keras')
    augmented_model_256.save('saved_models/augmented_model_batch_256.keras')
    augmented_model_384.save('saved_models/augmented_model_batch_384.keras')
    augmented_model_512.save('saved_models/augmented_model_batch_512.keras')
    augmented_model_1024.save('saved_models/augmented_model_batch_1024.keras')

In [ ]:
# read models and histories back for plotting
augmented_histories = {'Batch 128': None,
                       'Batch 256': None,
                       'Batch 384': None,
                       'Batch 512': None,
                       'Batch 1024': None}

# reading histories back
for batch_size in augmented_histories.keys():
    history_data = np.load(f'training_histories/augmented_history_{batch_size.replace(" ", "_")}.npy', allow_pickle=True).item()
    augmented_histories[batch_size] = history_data

# read models back
augmented_model_128 = keras.models.load_model('saved_models/augmented_model_batch_128.keras')
augmented_model_256 = keras.models.load_model('saved_models/augmented_model_batch_256.keras')
augmented_model_384 = keras.models.load_model('saved_models/augmented_model_batch_384.keras')
augmented_model_512 = keras.models.load_model('saved_models/augmented_model_batch_512.keras')
augmented_model_1024 = keras.models.load_model('saved_models/augmented_model_batch_1024.keras')

In [ ]:
# Evaluate and compare
test_loss_aug_128, test_acc_aug_128 = augmented_model_128.evaluate(X_test, y_test_encoded, verbose=0)
test_loss_aug_256, test_acc_aug_256 = augmented_model_256.evaluate(X_test, y_test_encoded, verbose=0)
test_loss_aug_384, test_acc_aug_384 = augmented_model_384.evaluate(X_test, y_test_encoded, verbose=0)
test_loss_aug_512, test_acc_aug_512 = augmented_model_512.evaluate(X_test, y_test_encoded, verbose=0)
test_loss_aug_1024, test_acc_aug_1024 = augmented_model_1024.evaluate(X_test, y_test_encoded, verbose=0)
# create a dictionary to store results
results = {
    'Batch 128': {'test_loss': test_loss_aug_128, 'test_acc': test_acc_aug_128},
    'Batch 256': {'test_loss': test_loss_aug_256, 'test_acc': test_acc_aug_256},
    'Batch 384': {'test_loss': test_loss_aug_384, 'test_acc': test_acc_aug_384},
    'Batch 512': {'test_loss': test_loss_aug_512, 'test_acc': test_acc_aug_512},
    'Batch 1024': {'test_loss': test_loss_aug_1024, 'test_acc': test_acc_aug_1024}
}

# calculate best test accuracy and corresponding batch size
best_model = None
best_test_acc = 0.0
for batch_size, metrics in results.items():
    if metrics['test_acc'] > best_test_acc:
        best_test_acc = metrics['test_acc']
        best_model = batch_size


print("\n" + "="*60)
print("CHALLENGER 1 RESULTS: Data Augmentation + Capacity") 
print("Batch sizes: 128, 256, 384, 512, 1024")
print("="*60)
for Batch_Size, history in augmented_histories.items():
    print(f"Batch Size: {Batch_Size.split(' ')[-1]}\n")
    print(f"Test Accuracy: {results[Batch_Size]['test_acc']*100:.2f}% (Baseline: {test_accuracy_baseline*100:.2f}%)")
    print(f"Improvement: {(results[Batch_Size]['test_acc'] - test_accuracy_baseline)*100:+.2f} percentage points") 
    print(f"Final Validation Accuracy: {history['val_accuracy'][-1]*100:.2f}%") 
    print(f"Best Validation Accuracy: {max(history['val_accuracy'])*100:.2f}%")
    print("-"*60)
print("="*60)
print(f"Overall Winner: {best_model} with best Test Accuracy: {best_test_acc*100:.2f}%")
print("="*60)


In [ ]:
# --- moving average helper (simple trailing window) ---
def moving_average(x, n=5):
    x = np.asarray(x, dtype=float)
    if n <= 1:
        return x
    if len(x) < n:
        # not enough points for a full window; return empty or fallback to original
        return np.array([])
    return np.convolve(x, np.ones(n) / n, mode="valid")

n = 5  # start with n=5

# mapping for history keys
key_map = {
    ("val",  "accuracy"): "val_accuracy",
    ("val",  "loss"):     "val_loss",
    ("test", "accuracy"): "accuracy",
    ("test", "loss"):     "loss",
}

# create 3x2 subplot layout:
# rows: test, val, val (moving average)
# cols: accuracy, loss
fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=False)

layout = [
    ("test", "accuracy", axes[0, 0], "raw"),
    ("test", "loss",     axes[0, 1], "raw"),
    ("val",  "accuracy", axes[1, 0], "raw"),
    ("val",  "loss",     axes[1, 1], "raw"),
    ("val",  "accuracy", axes[2, 0], "ma"),
    ("val",  "loss",     axes[2, 1], "ma"),
]

for row_name, metric_name, ax, mode in layout:
    hist_key = key_map[(row_name, metric_name)]

    for Batch_Size, history in augmented_histories.items():
        if hist_key not in history:
            continue

        y = history[hist_key]

        if mode == "ma":
            y_ma = moving_average(y, n=n)
            if y_ma.size == 0:
                continue
            # x shifts because MA(valid) shortens the series by n-1
            x_ma = np.arange(len(y_ma)) + (n - 1)
            ax.plot(x_ma, y_ma, label=f"{Batch_Size} (MA{n})", linewidth=1.8)
        else:
            ax.plot(y, label=f"{Batch_Size}", linewidth=1.5)

    ax.grid(alpha=0.3)

    if mode == "ma":
        ax.set_title(f"{row_name.title()} {metric_name.title()} (Moving Avg, n={n})",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("Epoch", fontsize=11)
    else:
        ax.set_title(f"{row_name.title()} {metric_name.title()}",
                     fontsize=12, fontweight="bold")

    if metric_name == "accuracy":
        ax.set_ylim(0, 1.0)
        ax.set_yticks(np.arange(0, 1.05, 0.05))
        ax.set_ylabel("Accuracy", fontsize=11)
    else:
        ax.set_ylabel("Loss", fontsize=11)

# X labels for the raw rows (test+val)
for ax in axes[1, :]:
    ax.set_xlabel("Epoch", fontsize=11)

# One legend for the whole figure
handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, title="Batch Size", loc="lower center", ncol=5, frameon=False)

fig.suptitle("Figure 5: Augmented Models: Test/Validation Accuracy & Loss (+ Val Moving Average)",
             fontsize=14, fontweight="bold", y=0.98)
fig.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.show()

### Analysis of Challenger 1

The train/val overfitting gap is substantially reduced, confirming that augmentation provides effective regularisation. Best test accuracy reaches ~89.8% — a ~25% improvement over baseline. Larger batch sizes (256→1024) show incremental gains, consistent with the hypothesis that stable gradient estimates benefit training on diverse augmented data. The model no longer exhibits the early plateau seen in the baseline, suggesting the augmented data diversity keeps the learning signal informative for more epochs.

**Decision**: Augmented model (batch=1024) becomes the new champion.


## 3.3 Challenger 2: Training Callbacks

Three callbacks are added to the champion architecture:

1. **ReduceLROnPlateau** – halves learning rate when validation loss plateaus, implementing a form of simulated annealing (Smith, 2017)
2. **EarlyStopping** – stops training when validation loss fails to improve for 10 epochs, acting as a regulariser by preventing over-training (Prechelt, 1998)
3. **ModelCheckpoint** – saves best validation weights during training, ensuring the returned model is the best seen rather than the final state


In [ ]:
# create callback-enhanced models (same architecture as Challenger 1)
callback_model = create_augmented_model(seed = 128, name='Callback_Enhanced_ConvNet')

# define callbacks
callbacks_list = [
    # early stopping: stop if val_loss doesn't improve for 10 epochs
    #EarlyStopping(
    #    monitor='val_loss',
    #    patience=20, # wait 20 epochs before stopping
    #   restore_best_weights=True,
    #    verbose=1
    #),
    
    # learning rate reduction: reduce LR when val_loss plateaus
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,           # reduce LR by half
        patience=10,           # wait 10 epochs before reducing
        min_lr=1e-8,          # don't go below this LR
        verbose=1
    ),
    
    # model checkpoint: save best model
    ModelCheckpoint(
        'best_model_weights.weights.h5',
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,
        verbose=0
    )
]

print("Callback Configuration:")
print("1. EarlyStopping: patience=10 epochs, restores best weights")
print("2. ReduceLROnPlateau: factor=0.5, patience=10 epochs")
print("3. ModelCheckpoint: saves best validation accuracy weights\n")

In [ ]:
# already ran and saved results
if False:
    # Train with callbacks
    print("Training Challenger 2: Data Augmentation + Callbacks")
    print("Training for up to 500 epochs (early stopping may terminate earlier)\n")

    history_callbacks = callback_model.fit(
        data_augmentation.flow(X_train, y_train_encoded, batch_size=1024),
        epochs=500,  # Increased max epochs; early stopping will terminate if appropriate
        validation_data=(X_val, y_val_encoded),
        callbacks=callbacks_list,
        verbose=1
    )

    # save history
    np.save('training_histories/callback_history.npy', history_callbacks.history)
    # save model weights (best weights are already saved by ModelCheckpoint)
    callback_model.save('saved_models/callback_enhanced_model.keras')

In [ ]:
# read history back
history_callbacks_data = np.load('training_histories/callback_history.npy', allow_pickle=True).item()
history_callbacks_saved = history_callbacks_data

# read model back
callback_model = keras.models.load_model('saved_models/callback_enhanced_model.keras')

In [ ]:
# Evaluate and compare
test_loss_cb, test_acc_cb = callback_model.evaluate(X_test, y_test_encoded, verbose=0)

print("\n" + "="*60)
print("CHALLENGER 2 RESULTS: Augmentation + Callbacks")
print("="*60)
print(f"Test Accuracy: {test_acc_cb*100:.2f}%")
print(f"Improvement over Baseline: {(test_acc_cb - test_accuracy_baseline)*100:+.2f} pp")
print(f"Improvement over Challenger 1: {(test_acc_cb - test_acc_aug_1024)*100:+.2f} pp")
print(f"\nTraining stopped at epoch: {len(history_callbacks_saved['loss'])}")
print(f"Best Validation Accuracy: {max(history_callbacks_saved['val_accuracy'])*100:.2f}%")
print(f"Best Epoch: {np.argmax(history_callbacks_saved['val_accuracy']) + 1}")
print("="*60)

# Plot training history
plot_training_history(history_callbacks_saved, 'Challenger 2: Model with Callbacks')

### Analysis of Challenger 2

ReduceLROnPlateau produced smoother convergence and achieved comparable accuracy in fewer epochs. ModelCheckpoint had negligible impact on final test accuracy, indicating that data augmentation already prevents the late-stage overfitting that checkpointing is designed to address — the dominant regularisation signal comes from augmentation, not early stopping.

**Decision**: Learning rate scheduling retained; EarlyStopping and ModelCheckpoint dropped for simplicity. Challenger 2 with ReduceLROnPlateau becomes the new champion.


## 3.4 Challenger 3: ResNet-Style Architecture with Batch Normalisation

**Motivation**: Plain sequential stacks suffer from (1) vanishing gradients in deeper layers, (2) the degradation problem where adding depth hurts accuracy, and (3) training instability from internal covariate shift. Residual connections (He et al., 2016) and Batch Normalisation (Ioffe & Szegedy, 2015) directly address all three.

**Residual connections**: Learn the residual mapping $F(x) = H(x) - x$ so the block output is `F(x) + x`. Skip connections provide direct gradient paths, enable identity mappings when no change is needed, and allow effective training of 50+ layer networks.

**Batch Normalisation**: Normalises mini-batch activations to zero mean / unit variance ($\hat{x} = (x - \mu_B) / \sqrt{\sigma_B^2 + \epsilon}$), reducing internal covariate shift, allowing higher learning rates, and providing mild stochastic regularisation.

**Architecture**: Scalable ResNet-style — initial Conv layer, three stages of *n* residual blocks (filters: 16→32→64), Global Average Pooling (replacing Flatten+Dense to reduce parameters), Dense(10, Softmax). Total depth = 1 + 6n conv layers. Projection shortcuts (1×1 conv) handle dimension changes between stages.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models


def residual_block(x, filters, kernel_size=3, stride=1, name=None):
    """
    Basic residual block (2x 3x3 convs) for CIFAR-style ResNet.

    x → Conv3x3(stride) → BN → ReLU → Conv3x3 → BN → (+) → ReLU
    |_______________________________________________________|
    """
    if name is None:
        name = "res_block"

    shortcut = x

    # main path
    x = layers.Conv2D(filters, kernel_size, strides=stride, padding="same",
                      use_bias=False, name=f"{name}_conv1")(x)
    x = layers.BatchNormalization(name=f"{name}_bn1")(x)
    x = layers.Activation("relu", name=f"{name}_relu1")(x)

    x = layers.Conv2D(filters, kernel_size, strides=1, padding="same",
                      use_bias=False, name=f"{name}_conv2")(x)
    x = layers.BatchNormalization(name=f"{name}_bn2")(x)

    # shortcut path
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding="same",
                                 use_bias=False, name=f"{name}_shortcut_conv")(shortcut)
        shortcut = layers.BatchNormalization(name=f"{name}_shortcut_bn")(shortcut)

    x = layers.Add(name=f"{name}_add")([x, shortcut])
    x = layers.Activation("relu", name=f"{name}_out")(x)
    return x


def create_resnet(input_shape=(32, 32, 3), num_classes=10, n=3, base_filters=16, custom_data_augmentation=True):
    """
    Scalable ResNet (He et al. 2015 style):

    - Stem: 3x3 conv, base_filters (default 16)
    - Stage 1 (32x32): n blocks, filters=16, stride=1
    - Stage 2 (16x16): n blocks, filters=32, first block stride=2
    - Stage 3 (8x8):  n blocks, filters=64, first block stride=2
    - GlobalAveragePooling
    - Dense softmax

    Total conv layers = 1 + 6n
      (1 stem conv) + (3 stages * n blocks/stage * 2 conv/block)
    """
    inputs = layers.Input(shape=input_shape, name="input")

    # data augmentation layer
    if custom_data_augmentation:
        # apply custom affine augmentation layer
        x = RandomAffineIDGLike(seed=128, name="custom_affine_aug")(inputs)
    else:
        x = inputs

    # stem conv 3x3, 16 filters
    x = layers.Conv2D(base_filters, 3, strides=1, padding="same",
                      use_bias=False, name="stem_conv")(x)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.Activation("relu", name="stem_relu")(x)

    # stage 1 filters=16, no downsampling
    filters = base_filters
    for i in range(n):
        x = residual_block(x, filters, stride=1, name=f"stage1_block{i+1}")

    # stage 2 filters=32, downsample at first block
    filters = base_filters * 2
    x = residual_block(x, filters, stride=2, name="stage2_block1")
    for i in range(1, n):
        x = residual_block(x, filters, stride=1, name=f"stage2_block{i+1}")

    # stage 3 filters=64, downsample at first block
    filters = base_filters * 4
    x = residual_block(x, filters, stride=2, name="stage3_block1")
    for i in range(1, n):
        x = residual_block(x, filters, stride=1, name=f"stage3_block{i+1}")

    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)

    # convert output to float32 for better numerical stability in softmax
    outputs = layers.Dense(num_classes, activation="softmax", name="output", dtype="float32")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name=f"resnet_n{n}")

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
        jit_compile=False, # disable XLA for better debugging and compatibility with custom layers
    )
    return model


# making models with n= 3, 5, 7, 9 blocks per stage and printing parameter counts:
for n in [3, 5, 7, 9]:
    m = create_resnet(n=n)
    print(m.name, "params:", f"{m.count_params():,}", "layers:", len(m.layers))
print("\nThe number of 'layers' includes all layers (conv, bn, add, activation, etc.)")

In [ ]:
# function to create callbacks for training
def make_callbacks(tag: str):
    return [
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1,
        ),
        ModelCheckpoint(
            filepath=f"checkpoints/best_resnet_{tag}.weights.h5",
            monitor="val_accuracy",
            save_best_only=True,
            save_weights_only=True,
            verbose=0,
        ),
    ]


# define true if you want to run this, but it took over 12 hours
if False:
    # loop to train ResNet models with different n values and save results
    for n in [ 9, 11]:
        # loop over batch sizes for each n
        for batch_size in [64, 128, 256, 384]:
            # create model
            model = create_resnet(n=n)
            # train model with data augmentation and callbacks
            history = model.fit(
                X_train, y_train_encoded,
                batch_size=batch_size,
                epochs=75,
                validation_data=(X_val, y_val_encoded),
                callbacks=make_callbacks(f"n{n}_batch{batch_size}"),
                verbose=1,
            )

            # save model and history
            model.save(f"saved_models/resnet_n{n}_batch{batch_size}.keras")
            np.save(f"training_histories/resnet_n{n}_batch{batch_size}.npy", history.history)

In [ ]:
# read models and histories back for evaluation and plotting
resnet_histories = {}
for n in [ 9, 11]:
    for batch_size in [64, 128, 256, 384]:
        tag = f"n{n}_batch{batch_size}"
        # read history
        history_data = np.load(f"training_histories/resnet_{tag}.npy", allow_pickle=True).item()
        resnet_histories[tag] = history_data
        # read model, with custom layer
        resnet_model = keras.models.load_model(f"saved_models/resnet_{tag}.keras", 
                                               custom_objects={"RandomAffineIDGLike": RandomAffineIDGLike})
        # evaluate on test set
        test_loss, test_acc = resnet_model.evaluate(X_test, y_test_encoded, verbose=0)
        print(f"ResNet n={n}, batch={batch_size} -> Test Accuracy: {test_acc*100:.2f}%")


In [ ]:
# visualize ResNet training histories
# create 4x2 subplot layout:
# rows: n=9-test, n=9-val, n=11-test, n=11-val
# cols: accuracy, loss

fig, axes = plt.subplots(4, 2, figsize=(14, 20), sharex=False)
layout = [
    ("n9", "test", "accuracy", axes[0, 0]),
    ("n9", "test", "loss", axes[0, 1]),
    ("n9", "val", "accuracy", axes[1, 0]),
    ("n9", "val", "loss", axes[1, 1]),
    ("n11", "test", "accuracy", axes[2, 0]),
    ("n11", "test", "loss", axes[2, 1]),
    ("n11", "val", "accuracy", axes[3, 0]),
    ("n11", "val", "loss", axes[3, 1]),
]

key_map = {
    ("test", "accuracy"): "accuracy",
    ("test", "loss"): "loss",
    ("val", "accuracy"): "val_accuracy",
    ("val", "loss"): "val_loss",
}

for n_tag, row_name, metric_name, ax in layout:
    hist_key = key_map[(row_name, metric_name)]

    # loop over batch sizes for this n and plot the metric
    for batch_size in [64, 128, 256, 384]:
        tag = f"{n_tag}_batch{batch_size}"
        history = resnet_histories[tag]

        y = history[hist_key]
        ax.plot(y, label=f"Batch {batch_size}", linewidth=1.5)

    ax.grid(alpha=0.3)

    if metric_name == "accuracy":
        ax.set_ylim(0.3, 1.0)
        ax.set_yticks(np.arange(0.3, 1.05, 0.05))
        ax.set_ylabel("Accuracy", fontsize=11)
    else:
        ax.set_ylabel("Loss", fontsize=11)

    if row_name == "val":
        ax.set_xlabel("Epoch", fontsize=11)

    ax.set_title(f"ResNet n={n_tag} {row_name.title()} {metric_name.title()}",
                 fontsize=12, fontweight="bold")
    
# One legend for the whole figure
handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, title="Batch Size", loc="lower center", ncol=4, frameon=False)

fig.suptitle("Figure 6: ResNet Models (n=9,11): Test/Validation Accuracy & Loss",
             fontsize=14, fontweight="bold", y=0.98)
fig.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.show()

### Analysis of Challenger 3

Test accuracy improves by ~1.8% over Challenger 2. Batch normalisation enables smoother convergence in roughly half the epochs, consistent with its role in reducing internal covariate shift. The train/val gap remains similar, suggesting the architectural change contributes primarily through representational capacity rather than additional regularisation. Global Average Pooling further reduces the parameter count compared to Flatten+Dense.

**Decision**: ResNet n=11, batch=64 is the **Section 3 champion** (~91.6% mean test accuracy).


## 3.5 Results and Comparison

Each model is evaluated with 5 random seeds to account for weight-initialisation stochasticity, providing mean ± std test accuracy estimates for reliable comparison.


In [ ]:
def evaluate_model_robust(model_fn, name, seeds=[128, 256, 512, 1024, 2048], 
                         use_augmentation=True, use_callbacks=True,
                         batch_size=1024):
    """
    Robustly evaluate a model across multiple random seeds.
    
    Args:
        model_fn: Function that creates and returns a compiled model
        name: Model name for display
        seeds: List of random seeds to use
        use_augmentation: Whether to use data augmentation
        use_callbacks: Whether to use training callbacks
        batch_size: Batch size to use for training (default 1024)
    
    Returns:
        Dictionary with mean and std of test accuracies
    """
    test_accuracies = []
    val_accuracies = []
    
    print(f"\nEvaluating {name} across {len(seeds)} random seeds...")
    print("-" * 60)
    
    for i, seed in enumerate(seeds):
        print(f"\nRun {i+1}/{len(seeds)} (seed={seed})")
        
        # set seeds
        np.random.seed(seed)
        tf.random.set_seed(seed)

        if model_fn.__name__ == "create_resnet":
            # for ResNet, pass n = 11
            model = model_fn(n=11)
        else:
            # create model
            model = model_fn(seed = seed)
        
        # prepare callbacks if needed
        if use_callbacks:
            callbacks = [
                # LR sceduler: reduce LR on plateau
                ReduceLROnPlateau(monitor='val_loss', factor=0.5, 
                                patience=5, min_lr=1e-7, verbose=0),
                # ModelCheckpoint to save best weights
                ModelCheckpoint(f'checkpoints/best_{name}_seed{seed}.weights.h5', 
                                monitor='val_accuracy',
                                save_best_only=True, 
                                save_weights_only=True,
                                verbose=0)
            ]
        else:
            callbacks = []
        
        # train
        if use_augmentation:
            history = model.fit(
                data_augmentation.flow(X_train, y_train_encoded, batch_size=batch_size),
                epochs=100,
                validation_data=(X_val, y_val_encoded),
                callbacks=callbacks,
                verbose=0
            )
        else:
            # only to be used with baseline model, which doesn't use augmentation, and uses batch_size=128
            history = model.fit(
                X_train, y_train_encoded,
                batch_size=batch_size,
                epochs=100,
                validation_data=(X_val, y_val_encoded),
                callbacks=callbacks,
                verbose=0
            )
        
        # evaluate
        _, test_acc = model.evaluate(X_test, y_test_encoded, verbose=0)
        test_accuracies.append(test_acc)
        val_accuracies.append(max(history.history['val_accuracy']))
        
        print(f"Test Accuracy: {test_acc*100:.2f}%")
    
    print("\n" + "=" * 60)
    print(f"RESULTS FOR {name.upper()}")
    print("=" * 60)
    print(f"Test Accuracy: {np.mean(test_accuracies)*100:.2f}% +- {np.std(test_accuracies)*100:.2f}%")
    print(f"Best Val Accuracy: {np.mean(val_accuracies)*100:.2f}% +- {np.std(val_accuracies)*100:.2f}%")
    print("=" * 60)
    
    return {
        'name': name,
        'test_mean': np.mean(test_accuracies),
        'test_std': np.std(test_accuracies),
        'val_mean': np.mean(val_accuracies),
        'val_std': np.std(val_accuracies),
        'test_accuracies': test_accuracies
    }

In [ ]:
# Evaluate all models robustly (using 5 seeds for demonstration)
# Note: In practice, you might use fewer seeds to save time during development

# this robust evaluation code is wrapped in an `if false:`
# because it takes ~2000 minutes to run (on my GPU).
if False:
    print("ROBUST MODEL EVALUATION")
    print("=" * 60)
    print("Evaluating each model across 5 random initializations")
    print("This may take some time...\n")

    results = []

    # Baseline
    results.append(evaluate_model_robust(
        create_baseline_model, 
        "Baseline ConvNet",
        use_augmentation=False,
        use_callbacks=False
    ))

    # Challenger 1: Augmentation + Capacity
    results.append(evaluate_model_robust(
        create_augmented_model,
        "Augmentation + Capacity",
        use_augmentation=True,
        use_callbacks=False
    ))

    # Challenger 2: Augmentation + Callbacks
    results.append(evaluate_model_robust(
        create_augmented_model,
        "Augmentation + Callbacks",
        use_augmentation=True,
        use_callbacks=True
    ))

    # Challenger 3: ResNet
    results.append(evaluate_model_robust(
        create_resnet,
        "ResNet + BatchNorm",
        use_augmentation=False,  # ResNet already has its own custom augmentation layer
        use_callbacks=True,
        batch_size=64  # ResNet models were found to be best with smaller batch size during development
    ))

    # save all results 
    np.save('evaluations/robust_evaluation_results.npy', results)

In [ ]:
# read results back
robust_results = np.load('evaluations/robust_evaluation_results.npy', allow_pickle=True)

In [ ]:
# Create comprehensive comparison table
comparison_df = pd.DataFrame([
    {
        'Model': r['name'],
        'Test Accuracy (%)': f"{r['test_mean']*100:.2f} ± {r['test_std']*100:.2f}",
        'Best Val Accuracy (%)': f"{r['val_mean']*100:.2f} ± {r['val_std']*100:.2f}",
        'Mean Test Acc': r['test_mean']
    }
    for r in robust_results
])

# Sort by test accuracy
comparison_df = comparison_df.sort_values('Mean Test Acc', ascending=False)
comparison_df = comparison_df.drop('Mean Test Acc', axis=1)
comparison_df = comparison_df.reset_index(drop=True)

print("\n" + "=" * 70)
print("COMPREHENSIVE MODEL COMPARISON")
print("=" * 70)
print(comparison_df.to_string(index=False))
print("=" * 70)
print("\nNote: Values shown as mean ± standard deviation across 5 runs")
print("Higher mean indicates better performance")
print("Lower std indicates more stable/reproducible performance")

In [ ]:
# Visualize results comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot with error bars
model_names = [r['name'] for r in robust_results]
test_means = [r['test_mean'] * 100 for r in robust_results]
test_stds = [r['test_std'] * 100 for r in robust_results]

x_pos = np.arange(len(model_names))
ax1.bar(x_pos, test_means, yerr=test_stds, capsize=5, alpha=0.7, 
        color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
ax1.set_xlabel('Model', fontsize=11)
ax1.set_ylabel('Test Accuracy (%)', fontsize=11)
ax1.set_title('Model Performance Comparison (Mean ± Std)', fontsize=12, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(model_names, rotation=15, ha='center')
ax1.grid(axis='y', alpha=0.3)
# add value labels on top of bars
for i, (mean, std) in enumerate(zip(test_means, test_stds)):
    ax1.text(i, mean - std - 5, f"{mean:.2f}±{std:.2f}%", ha='center', va='bottom', fontsize=9)


# Box plot showing distribution
test_acc_data = [np.array(r['test_accuracies']) * 100 for r in robust_results]
bp = ax2.boxplot(test_acc_data, tick_labels=model_names, patch_artist=True)
for patch, color in zip(bp['boxes'], ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax2.set_xlabel('Model', fontsize=11)
ax2.set_ylabel('Test Accuracy (%)', fontsize=11)
ax2.set_title('Test Accuracy Distribution Across Seeds', fontsize=12, fontweight='bold')
ax2.set_xticklabels(model_names, rotation=15, ha='center', )
ax2.grid(axis='y', alpha=0.3)
for i, (mean, std) in enumerate(zip(test_means, test_stds)):
    ax2.text(i + 1, mean + std + 0.5, f"{mean:.2f}±{std:.2f}%", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

### Discussion of Results

| Feature | Approx. Gain | Primary Benefit |
|---------|-------------|------------------|
| Convolutional baseline (§2) | +20% over dense | Spatial feature learning |
| Augmentation + capacity (§3.2) | +17% | Regularisation, effective data expansion |
| Training callbacks (§3.3) | +0.5% | Optimal LR schedule |
| ResNet + Batch Norm (§3.4) | +5% | Faster convergence, deeper networks |

Data augmentation dominates, reinforcing that data quality and diversity are the primary bottleneck at this scale. Architecture improvements provide meaningful but secondary gains — sophisticated architectures can only reach their potential when the training data is sufficiently rich and varied. Training callbacks provided marginal improvement on top of augmentation, confirming that augmentation itself acts as the primary regulariser. The robust multi-seed evaluation confirms that all reported differences exceed run-to-run variance and represent genuine improvements.


---

# 4 | Model development – using a pretrained model with fine tuning

## 4.1 | Why pretrained models?

Pretrained CNNs trained on ImageNet encode hierarchical visual features (edges → textures → shapes → object parts) that transfer well across vision tasks. Fine-tuning adapts these generic representations to CIFAR-10 with far fewer gradient updates than training from random initialisation.

**Key constraint**: ImageNet models expect 224×224 inputs; CIFAR-10's 32×32 images must be upsampled. Input preprocessing (mean subtraction, scaling) must also match each model's expected statistics. Early backbone layers (generic features) should be kept frozen during initial fine-tuning to avoid destroying useful pretrained representations.

**Fine-tuning schedule**: Stage A — train new classification head only (base frozen, LR ~1e-3); Stage B — unfreeze top backbone layers and fine-tune at a reduced LR (~1e-5).

> **Citations**: EfficientNet: Tan & Le (2019). ResNet50V2: He et al. (2016). Keras Applications: Chollet et al.


## 4.2 | Preprocessing + resizing pipeline for ImageNet models

We resize CIFAR-10 from **32×32 -> 224×224** and apply the model-specific `preprocess_input`.


In [ ]:
# re-load cifar data
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.cifar10.load_data()

# split train into train and val
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1, random_state=128)\

# one-hot encode labels
num_classes = 10
y_train_encoded = keras.utils.to_categorical(y_train, num_classes)
y_val_encoded = keras.utils.to_categorical(y_val, num_classes)
y_test_encoded = keras.utils.to_categorical(y_test, num_classes)

In [ ]:
from tensorflow.keras.applications import EfficientNetB0, ResNet50V2
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess
from tensorflow.keras.applications.resnet_v2 import preprocess_input as resnetv2_preprocess

def build_pretrained_model(
    base_name='efficientnetb0',
    num_classes=10,
    dropout=0.2,
    dense_units=256,
    train_base=False,
    fine_tune_at=None,
    ):
    """
    Builds a transfer learning model with correct preprocessing.

    Args:
      base_name: 'efficientnetb0' or 'resnet50v2'
      train_base: whether the base is trainable
      fine_tune_at: the layer index to start fine-tuning
    """
    
    # select base model and preprocessing function based on base_name
    # if efficientnetb0
    if base_name.lower() == 'efficientnetb0':
        base = EfficientNetB0(
            include_top=False,
            weights='imagenet',
            input_shape=(224, 224, 3),
        )
        preprocess = effnet_preprocess
        base_tag = 'EfficientNetB0'
    
    # if resnet50v2
    elif base_name.lower() == 'resnet50v2':
        base = ResNet50V2(
            include_top=False,
            weights='imagenet',
            input_shape=(224, 224, 3),
        )
        preprocess = resnetv2_preprocess
        base_tag = 'ResNet50V2'
    else:
        raise ValueError("Unknown base_name")


    # set trainability of the base model and optionally freeze layers for fine-tuning
    base.trainable = train_base
    if train_base and fine_tune_at is not None:
        for layer in base.layers[:fine_tune_at]:
            layer.trainable = False

    # build the model with augmentation, preprocessing, base, and custom head
    inputs = layers.Input(shape=(32, 32, 3), name='input_32')

    # resize
    x = layers.Resizing(224, 224, name='resize')(inputs)
    
    # preprocess for the chosen backbone
    # takes raw images in [0, 255]
    x = layers.Lambda(preprocess, name='preprocess_input')(x)

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(dropout, name='dropout')(x)
    x = layers.Dense(dense_units, activation='relu', name='dense')(x)
    x = layers.Dropout(dropout, name='dropout2')(x)

    # force float32 for stable softmax under mixed precision
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32', name='output')(x)

    model = models.Model(inputs, outputs, name=f'TL_{base_tag}')
    return model


def compile_for_stage(model, lr):
    opt = keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    return model


print("Pretrained model builders ready")

## 4.3 | Classification head design

The `build_pretrained_model` head: Global Average Pooling → BatchNorm → Dropout(0.4) → Dense(256, ReLU) → Dense(10, Softmax). Stage A trains only this head; Stage B also updates the top backbone layers at a reduced learning rate (~1e-5).


## 4.4 | Training protocol

We implement a helper that trains:

- **Stage A** (Frozen base): LR ~ 1e-2, fewer epochs.
- **Stage B** (Fine-tune top of base): LR ~ 1e-3 to 1e-5.


In [ ]:
def train_transfer_learning(
    base_name,
    batch_size=64,
    epochs_frozen=15,
    epochs_finetune=20,
    lr_frozen=1e-2,
    lr_finetune=1e-3,
    fine_tune_at=None,
    ):
    """Train transfer learning model with a two-stage schedule."""

    # Stage A: frozen base
    model = build_pretrained_model(
        base_name=base_name,
        train_base=False,
    )
    compile_for_stage(model, lr=lr_frozen)

    cbA = make_callbacks(tag=f'{model.name}_stageA')
    # train with frozen base
    histA = model.fit(
        data_augmentation.flow(X_train, y_train_encoded, batch_size=batch_size),
        epochs=epochs_frozen,
        validation_data=(X_val, y_val_encoded),
        callbacks=cbA,
        verbose=1,
    )

    # Stage B: fine-tune top layers
    # rebuild model but now allow base to train
    model_ft = build_pretrained_model(
        base_name=base_name,
        train_base=True,
        fine_tune_at=fine_tune_at,
    )

    # load weights from Stage A so fine-tuning starts from that point
    model_ft.set_weights(model.get_weights())
    compile_for_stage(model_ft, lr=lr_finetune)

    cbB = make_callbacks(tag=f'{model_ft.name}_stageB')
    histB = model_ft.fit(
        data_augmentation.flow(X_train, y_train_encoded, batch_size=batch_size),
        epochs=epochs_finetune,
        validation_data=(X_val, y_val_encoded),
        callbacks=cbB,
        verbose=1,
    )

    # test evaluation
    test_loss, test_acc = model_ft.evaluate(X_test, y_test_encoded, verbose=0)
    return model_ft, histA.history, histB.history, (test_loss, test_acc)


print("Training helper ready")

## 4.5 | Run pretrained experiments

The `fine_tune_at` index controls how much of the base is frozen during fine-tuning. Lower values = unfreeze more layers. 

In [ ]:
# =============================
# Pretrained training runs
# =============================

os.makedirs('training_histories', exist_ok=True)
os.makedirs('saved_models', exist_ok=True)

if False:
    # EfficientNetB0
    eff_model, eff_histA, eff_histB, (eff_test_loss, eff_test_acc) = train_transfer_learning(
        base_name='efficientnetb0',
        batch_size=64,
        epochs_frozen=35,
        epochs_finetune=35,
        lr_frozen=1e-3,
        lr_finetune=1e-5,
        fine_tune_at=200,
    )
    eff_model.save('saved_models/tl_efficientnetb0.keras')
    np.save('training_histories/tl_eff_stageA.npy', eff_histA)
    np.save('training_histories/tl_eff_stageB.npy', eff_histB)
    np.save('training_histories/tl_eff_test.npy', {'loss': eff_test_loss, 'acc': eff_test_acc})

    # ResNet50V2
    res_model, res_histA, res_histB, (res_test_loss, res_test_acc) = train_transfer_learning(
        base_name='resnet50v2',
        batch_size=64,
        epochs_frozen=35,
        epochs_finetune=25,
        lr_frozen=1e-3,
        lr_finetune=1e-5,
        fine_tune_at=120,
    )
    res_model.save('saved_models/tl_resnet50v2.keras')
    np.save('training_histories/tl_res_stageA.npy', res_histA)
    np.save('training_histories/tl_res_stageB.npy', res_histB)
    np.save('training_histories/tl_res_test.npy', {'loss': res_test_loss, 'acc': res_test_acc})

## 4.6 | Load pretrained results

Since training can take a while, the results have been saved and can be loaded directly for analysis.

In [ ]:
def safe_load_npy(path):
    if os.path.exists(path):
        return np.load(path, allow_pickle=True).item()
    return None


eff_histA = safe_load_npy('training_histories/tl_eff_stageA.npy')
eff_histB = safe_load_npy('training_histories/tl_eff_stageB.npy')
eff_test = safe_load_npy('training_histories/tl_eff_test.npy')

res_histA = safe_load_npy('training_histories/tl_res_stageA.npy')
res_histB = safe_load_npy('training_histories/tl_res_stageB.npy')
res_test = safe_load_npy('training_histories/tl_res_test.npy')

print("Loaded:")
print("EfficientNet histories:", eff_histA is not None, eff_histB is not None, "test:", eff_test)
print("ResNet50V2 histories:", res_histA is not None, res_histB is not None, "test:", res_test)

In [ ]:
def plot_history_pair(histA, histB, title):
    if histA is None or histB is None:
        print("No history found for", title)
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    
    # Stage A
    axes[0,0].plot(histA['accuracy'], label='train')
    axes[0,0].plot(histA['val_accuracy'], label='val')
    axes[0,0].set_title('Stage A (frozen base): accuracy')
    axes[0,0].grid(alpha=0.3)
    axes[0,0].legend()
    
    axes[0,1].plot(histA['loss'], label='train')
    axes[0,1].plot(histA['val_loss'], label='val')
    axes[0,1].set_title('Stage A (frozen base): loss')
    axes[0,1].grid(alpha=0.3)
    axes[0,1].legend()
    
    # Stage B
    axes[1,0].plot(histB['accuracy'], label='train')
    axes[1,0].plot(histB['val_accuracy'], label='val')
    axes[1,0].set_title('Stage B (fine-tune): accuracy')
    axes[1,0].grid(alpha=0.3)
    axes[1,0].legend()
    
    axes[1,1].plot(histB['loss'], label='train')
    axes[1,1].plot(histB['val_loss'], label='val')
    axes[1,1].set_title('Stage B (fine-tune): loss')
    axes[1,1].grid(alpha=0.3)
    axes[1,1].legend()

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


plot_history_pair(eff_histA, eff_histB, 'Figure 7: EfficientNetB0 Transfer Learning (Stage A + B)')
plot_history_pair(res_histA, res_histB, 'Figure 8: ResNet50V2 Transfer Learning (Stage A + B)')

## 4.7 | Pretrained results table + comparison with Section 3 champion

Your Section 3 robust table indicated a strong champion around ~**89–90%**. Transfer learning may exceed this, match it, or underperform (depending on fine-tuning schedule and the domain shift CIFAR→ImageNet).

The table below is populated if you have saved pretrained results.

In [ ]:
rows = []
if eff_test is not None:
    rows.append({'Model': 'TL EfficientNetB0', 'Test Accuracy (%)': 100*eff_test['acc'], 'Test Loss': eff_test['loss']})
if res_test is not None:
    rows.append({'Model': 'TL ResNet50V2', 'Test Accuracy (%)': 100*res_test['acc'], 'Test Loss': res_test['loss']})

if len(rows) == 0:
    print("No pretrained results saved yet. Run the training cell in Section 4.5.")
else:
    df = pd.DataFrame(rows).sort_values('Test Accuracy (%)', ascending=False)
    display(df)


Fine-tuned pretrained models substantially outperform the Section 3 champion (~91.6%). ResNet50V2 achieves ~95.7% test accuracy (+4.1%), benefiting from both its ImageNet-pretrained weights and the strong architectural similarity to our custom ResNet. EfficientNetB0 also performs well (~93%) but is limited by its smaller parameter count under the compound-scaling design philosophy. Both results are saved from `tl_res_test.npy` and `tl_eff_test.npy` respectively.


---

# 5 | The best model

## 5.1 | Selection criteria and decision

Models are ranked on four criteria: test accuracy (primary), stability across seeds, compute cost, and conceptual justification.

| Model | Test Acc. | Notes |
|-------|-----------|-------|
| Baseline ConvNet | ~65% | Overfits; no regularisation |
| Augmented ConvNet (Ch1, batch-1024) | ~89.8% | S3 stepping stone |
| ResNet+BN n=11 (Ch3, batch-64) | ~91.6% | S3 champion |
| TL EfficientNetB0 | ~93% | Pretrained, compact |
| TL ResNet50V2 | ~95.7% | **Best overall** |

**Best model: pretrained ResNet50V2.** It achieves the highest test accuracy with strong stability from its ImageNet-pretrained weights. The architectural alignment with our custom ResNet (residual blocks, similar depth philosophy) makes the feature transfer particularly effective. The compute cost (two-stage fine-tuning) is acceptable given the substantial accuracy gain.


## 5.2 | Best model training function

`best_model_fn` / `train_and_eval_best` wrap the pretrained ResNet50V2 pipeline (Stage A + Stage B fine-tuning) for reproducible single-run or ensemble training.


In [ ]:
def best_model_fn():
    """Return a compiled best model.
    """
    model = build_pretrained_model('resnet50v2', train_base=True, fine_tune_at=120)
    compile_for_stage(model, lr=1e-5)
    return model


def train_and_eval_best(seed=128, batch_size=64, epochs=30):
    tf.keras.utils.set_random_seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    # train for transfer learning
    model_ft, histA, histB, (test_loss, test_acc) = train_transfer_learning(
        base_name='resnet50v2',
        batch_size=batch_size,
        epochs_frozen=epochs,
        epochs_finetune=epochs,
        lr_frozen=1e-3,
        lr_finetune=1e-5,
        fine_tune_at=120,
    )

    
    return model_ft, histA, histB, (test_loss, test_acc)


print("Best-model training helpers ready")

## 5.3 | Ensembling (variance reduction)

Even a well-tuned single model has stochastic variance from weight initialisation, augmentation, and mini-batch ordering. Ensembling K independently trained models (different seeds) and averaging their softmax outputs reduces this variance, consistently providing a modest accuracy boost over any single model. Training and evaluation code is gated behind `if False:` (3 seeds used by default; increase for tighter variance estimates).


In [ ]:
def ensemble_predict(models_list, X):
    probs = [m.predict(X, verbose=0) for m in models_list]
    mean_probs = np.mean(probs, axis=0)
    return mean_probs


def accuracy_from_probs(probs, y_true_int):
    y_pred = np.argmax(probs, axis=1)
    return np.mean(y_pred == y_true_int.flatten())


if False:
    seeds = [128, 256, 512]

    trained_models = []
    histories = []
    for s in seeds:
        m, hA, hB, (tl, ta) = train_and_eval_best(seed=s, batch_size=64, epochs=25)
        trained_models.append(m)
        histories.append((hA, hB))
        print(f"Seed {s}: test acc = {ta*100:.2f}%")

    probs = ensemble_predict(trained_models, X_test)
    ens_acc = accuracy_from_probs(probs, y_test)
    print(f"\nEnsemble accuracy ({len(seeds)} models): {ens_acc*100:.2f}%")

In [ ]:
# save the ensemble results
if False:
    np.save('evaluations/ensemble_results.npy', {
        'seeds': seeds,
        'test_accuracies': [h[1]['val_accuracy'][-1] for h in histories],  # final val acc from each model
        'ensemble_accuracy': ens_acc,
    })

# read ensemble results back
ensemble_results = np.load('evaluations/ensemble_results.npy', allow_pickle=True).item()
print("Ensemble evaluation results loaded:")
print(f"Seeds: {ensemble_results['seeds']}")
print(f"Individual model test accuracies: {[f'{acc*100:.2f}%' for acc in ensemble_results['test_accuracies']]}")
print(f"Ensemble accuracy: {ensemble_results['ensemble_accuracy']*100:.2f}%")
print(f"Greater than the sum of its parts")

---

# 6 | Visualisation of the model

This section visualises the **augmented ConvNet (Challenger 1, batch-1024)**, loaded from `saved_models/augmented_model_batch_1024.keras`. This model is used rather than ResNet50V2 because it operates on native 32×32 inputs, making filters and activations more interpretable.

Three visualisations: (1) first-layer **convolution kernels**, (2) **intermediate activations** at early/mid/deep layers, (3) **Grad-CAM** heatmaps (Selvaraju et al., 2017).


In [ ]:
Model_Path = 'saved_models/augmented_model_batch_1024.keras'
viz_model = keras.models.load_model(
    Model_Path,
    custom_objects={"RandomAffineIDGLike": RandomAffineIDGLike},
    compile=False,   # recommended for viz-only
    safe_mode=False  # sometimes needed with custom objects
)

# Build/call once so model.output exists
_ = viz_model.predict(X_test[:1].astype("float32"), verbose=0)

viz_model.summary()

## 6.1 | Visualise early convolution kernels

Early filters typically correspond to simple edge and colour detectors.


In [ ]:
def find_first_conv2d(model):
    for l in model.layers:
        if isinstance(l, layers.Conv2D):
            return l
    return None


def plot_conv_kernels(conv_layer, max_kernels=36):
    weights = conv_layer.get_weights()
    if not weights:
        raise ValueError(f"Layer {conv_layer.name} has no weights")

    k = weights[0]  # (kH, kW, inC, outC)
    kH, kW, inC, outC = k.shape
    n = min(outC, max_kernels)
    grid = int(math.ceil(math.sqrt(n)))

    fig, axes = plt.subplots(grid, grid, figsize=(10, 10))
    axes = np.array(axes).reshape(-1)
    for i in range(grid * grid):
        ax = axes[i]
        ax.axis('off')
        if i >= n:
            continue

        ker = k[:, :, :, i]
        ker = (ker - ker.min()) / (ker.max() - ker.min() + 1e-8)

        if inC >= 3:
            ax.imshow(ker[:, :, :3])
        else:
            ax.imshow(ker[:, :, 0], cmap='viridis')

    fig.suptitle(f"Figure 6.1: First-layer kernels ({conv_layer.name})", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


first_conv = find_first_conv2d(viz_model)
if first_conv is None:
    print('No Conv2D found in top-level model layers. (If your model is nested, adjust the search.)')
else:
    plot_conv_kernels(first_conv)


## 6.2 | Visualise intermediate activations (feature maps)

We extract feature maps from early, mid, and late convolution layers.


In [ ]:
def list_conv_layer_names(model):
    return [l.name for l in model.layers if isinstance(l, layers.Conv2D)]


def show_feature_maps(model, layer_name, image, max_channels=36):
    img = image.astype('float32')
    if img.max() > 1.5:
        img = img / 255.0

    feat_model = keras.Model(inputs=model.inputs, outputs=model.get_layer(layer_name).output)
    activ = feat_model.predict(img[None, ...], verbose=0)[0]  # (H,W,C)

    H, W, C = activ.shape
    n = min(C, max_channels)
    grid = int(math.ceil(math.sqrt(n)))

    fig, axes = plt.subplots(grid, grid, figsize=(10, 10))
    axes = np.array(axes).reshape(-1)
    for i in range(grid * grid):
        ax = axes[i]
        ax.axis('off')
        if i >= n:
            continue
        fmap = activ[:, :, i]
        fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min() + 1e-8)
        ax.imshow(fmap, cmap='viridis')

    fig.suptitle(f"Figure 6.2: Feature maps from layer '{layer_name}'", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


conv_names = list_conv_layer_names(viz_model)
print('Number of Conv2D layers:', len(conv_names))
print('First 10 conv layers:', conv_names[:10])

early = conv_names[0]
mid = conv_names[len(conv_names)//2]
late = conv_names[-1]
chosen = [early, mid, late]
print('Chosen layers:', chosen)

idx = 128
plt.figure(figsize=(3,3))
plt.imshow(X_test[idx])
plt.title(f"Input image (true: {class_names[int(y_test[idx][0])]})")
plt.axis('off')
plt.show()

for ln in chosen:
    show_feature_maps(viz_model, ln, X_test[idx], max_channels=36)


## 6.3 | Grad-CAM

Grad-CAM produces a heatmap highlighting spatial regions contributing to the prediction.


In [ ]:
def find_last_conv2d_name(model):
    last = None
    for l in model.layers:
        if isinstance(l, layers.Conv2D):
            last = l.name
    return last


def make_gradcam_heatmap_sequential_like(model, img, last_conv_layer_name, pred_index=None):
    img = img.astype("float32")
    if img.max() > 1.5:
        img = img / 255.0
    x0 = tf.convert_to_tensor(img[None, ...])

    # Ensure the model is actually called in eager mode (stronger than predict sometimes)
    _ = model(x0, training=False)

    # Split model into: [input -> last_conv] and [last_conv -> output]
    conv_idx = None
    for i, l in enumerate(model.layers):
        if l.name == last_conv_layer_name:
            conv_idx = i
            break
    if conv_idx is None:
        raise ValueError(f"Layer not found: {last_conv_layer_name}")

    conv_layers = model.layers[: conv_idx + 1]
    head_layers = model.layers[conv_idx + 1 :]

    with tf.GradientTape() as tape:
        x = x0
        # forward to conv feature maps
        for l in conv_layers:
            x = l(x, training=False)
        conv_out = x
        tape.watch(conv_out)

        # forward through head to predictions
        for l in head_layers:
            x = l(x, training=False)
        preds = x

        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_score = preds[:, pred_index]

    grads = tape.gradient(class_score, conv_out)
    if grads is None:
        raise ValueError(
            "Grad-CAM failed: gradients are None. Try using an earlier conv layer "
            "(e.g. 'conv3_2' or 'conv2_3')."
        )

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))  # (C,)
    conv_out = conv_out[0]  # (H,W,C)

    heatmap = tf.reduce_sum(conv_out * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)


def overlay_heatmap(img, heatmap, alpha=0.45):
    heatmap = tf.image.resize(heatmap[..., None], (img.shape[0], img.shape[1]))[..., 0].numpy()
    cmap = plt.get_cmap("jet")
    colored = cmap(heatmap)[:, :, :3]

    img = img.astype("float32")
    if img.max() > 1.5:
        img = img / 255.0

    overlay = (1 - alpha) * img + alpha * colored
    return np.clip(overlay, 0, 1)


# pick last conv layer
last_conv = find_last_conv2d_name(viz_model)
print("Using last conv layer:", last_conv)

# choose an image
idx = 37
img = X_test[idx]
true_class = int(y_test[idx][0])

# normal prediction (also ensures the model is callable)
preds = viz_model(img[None, ...].astype("float32"), training=False).numpy()
pred_class = int(np.argmax(preds[0]))

# Grad-CAM
heatmap, _ = make_gradcam_heatmap_sequential_like(viz_model, img, last_conv, pred_index=pred_class)
overlay = overlay_heatmap(img, heatmap)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img)
axes[0].set_title(f"Original\nTrue: {class_names[true_class]}")
axes[0].axis("off")

axes[1].imshow(heatmap, cmap="jet")
axes[1].set_title(f"Grad-CAM\nPred: {class_names[pred_class]}")
axes[1].axis("off")

axes[2].imshow(overlay)
axes[2].set_title("Overlay")
axes[2].axis("off")

fig.suptitle("Figure 6.3: Grad-CAM visualisation (Augmented ConvNet, batch=1024)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---

# 7 | Discussion

## 7.1 | Summary of findings

| Stage | Key result |
|-------|------------|
| Baseline (§2) | ~65% test acc; clear overfitting (~18% train/val gap) |
| Augmentation + capacity (§3.2) | ~89.8%; overfitting gap largely closed |
| ResNet + BN, n=11 (§3.4) | ~91.6%; faster convergence, lower parameter count |
| TL ResNet50V2 (§4) | ~95.7%; strongest result; pretrained features dominant |

The cumulative improvement from ~65% to ~95.7% demonstrates the compounding value of a systematic development pipeline. Each stage addresses a specific limitation of the previous champion.

## 7.2 | Technique contributions

- **Data augmentation**: single largest gain (~17%); reduced overfitting by expanding the effective training set and forcing invariance learning.
- **Residual connections**: enabled stable training of deeper networks; improved gradient flow by providing skip paths.
- **Batch normalisation**: accelerated convergence (fewer epochs), mild stochastic regularisation.
- **Transfer learning**: dominant improvement overall; ImageNet features transfer well even under the 32×32 → 224×224 upsampling constraint.
- **Ensembling**: reduced prediction variance across seeds, yielding modest accuracy gains at inference.

## 7.3 | Visualisation discussion

- **Early kernels**: appear noisy rather than showing clean edge/colour detectors. This is expected for stacked 3×3 convolutions on 32×32 images: discriminative information is distributed across many non-linear combinations of filters rather than concentrated in single visually interpretable templates.
- **Intermediate activations**: early layers produce dense, spatially broad activations (texture/edge detectors); mid and deep layers become progressively sparser and more class-selective, consistent with hierarchical feature learning.
- **Grad-CAM**: heatmaps consistently focus on the foreground object (e.g. body of a vehicle, silhouette of an animal), confirming the model uses semantically meaningful spatial cues. Failure cases — heatmaps focusing on background regions — highlight potential shortcut learning and dataset biases worth addressing in future work.

## 7.4 | Limitations and future work

- **Resolution mismatch**: CIFAR-10's 32×32 inputs require upsampling to 224×224 for pretrained models, introducing artefacts and significant compute overhead.
- **Augmentation quality**: GPU Keras augmentation layers introduced blurring (see Appendix A and `data_aug_testing.ipynb`); CPU `ImageDataGenerator` was preferred for image fidelity.
- **Batch size sensitivity**: optimal batch size differed between architectures (larger for ConvNet, smaller for ResNet), suggesting architecture-specific hyperparameter search is needed.
- **Ensemble size**: only 3 seeds due to compute constraints; larger ensembles (≥10) would give tighter variance estimates.
- **Future work**: CutMix/MixUp augmentation, cosine-annealing LR schedules, and knowledge distillation from ResNet50V2 to a compact student model deployable at native 32×32 resolution.


# Appendix A: Technical Reflections

#### Data Augmentation Bottleneck

- Ran into training bottleneck
- Data augmentation would cause training to freeze
- Data augmentation was taking too long to compute, as training was performed on GPU
- Switched to `keras.layers.Random[transform]` for efficient data augmentation using GPU
- Some of the problem was due to using WSL, because I/O performance is poor from the WSL filesystem,
- Tried using keras transform layers for data aug, but results were much blurier than the CPU bound ImageDataGenerator.
- This is likely due to the GPU method performing interpolation between each transform and the CPU method performing one combined transform, and a single interpolation after
- Investigated this and found that the CPU augmentation indeed distorted the images less than keras layers (see data_aug_testing.ipynb)
- Chose to use CPU augmentation for better performance, despite the bottleneck, as it provided much better results than GPU augmentation


#### ResNet Batch Sizes

- Ran into accidental small batch size of 128, which is smaller than the intended batch size 1024 found previously for non-residual models
- Reversed this accident to 1024, which provided worse results than 128
- This directly contradicts the previous findings on batch sizes, where the range 128-1024 provided sequentially better converged model validation accuracy, with 1024 providing the best results.
- This suggests that the optimal batch size may depend on the architecture and training dynamics

---

## Word Count Guidance

The marking rubric requires that the report body does not exceed **3,000 words** (excluding code, tables, plots/figures, and references). Run the optional code cell below to get an estimate of the current Markdown word count. Do not include code cells or figure captions in the word count.


In [ ]:
# OPTIONAL – word count estimator for Markdown cells
# This cell is NOT run by default; remove the `if False:` to execute.
if False:
    import re, json
    with open('CIFAR10_Neural_Networks_Coursework_REVISED.ipynb') as _f:
        _nb = json.load(_f)
    _words = []
    for _c in _nb['cells']:
        if _c['cell_type'] == 'markdown':
            _text = ''.join(_c['source'])
            # strip markdown syntax, code fences, and HTML tags
            _text = re.sub(r'```.*?```', '', _text, flags=re.DOTALL)
            _text = re.sub(r'<[^>]+>', '', _text)
            _text = re.sub(r'[#*`_>\[\]|]', ' ', _text)
            _words.extend(_text.split())
    print(f'Estimated Markdown word count: {len(_words)}')
    print(f'Limit: 3000 words')
    if len(_words) > 3000:
        print('WARNING: word count exceeds 3000 – please trim.')
    else:
        print('Word count within limit.')


---

## References

- Chollet, F. et al. (2015). *Keras*. Available at: https://keras.io
- He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. *Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR)*, pp. 770–778.
- He, K., Zhang, X., Ren, S., & Sun, J. (2016). Identity mappings in deep residual networks. *European Conference on Computer Vision (ECCV)*, pp. 630–645. *(ResNet50V2 variant)*
- Krizhevsky, A., & Hinton, G. (2009). *Learning Multiple Layers of Features from Tiny Images*. Technical Report, University of Toronto.
- Selvaraju, R.R., Cogswell, M., Das, A., Vedantam, R., Parikh, D., & Batra, D. (2017). Grad-CAM: Visual explanations from deep networks via gradient-based localization. *Proceedings of the IEEE International Conference on Computer Vision (ICCV)*, pp. 618–626.
- Tan, M., & Le, Q. (2019). EfficientNet: Rethinking model scaling for convolutional neural networks. *Proceedings of the 36th International Conference on Machine Learning (ICML)*, pp. 6105–6114.
